# OCR Evaluation — единый Kaggle-пайплайн

Сравнение трёх OCR-моделей на OmniDocBench (страницы `academic_literature`):

1. **LightOnOCR-2-1B** — `lightonai/LightOnOCR-2-1B`, компактная, быстрая.
2. **DeepSeek-OCR** — `deepseek-ai/DeepSeek-OCR`, ~3B, grounding-промпт.
3. **olmOCR-7B** — `allenai/olmOCR-7B-0225-preview`, Qwen2-VL-7B fine-tuned.

Ноутбук рассчитан на Kaggle с одной GPU **Tesla P100** (16 ГБ).
Все шаги в одном файле: setup → датасет → инференс трёх моделей → быстрая верификация.

> Перед запуском в Kaggle: `Add-ons → Secrets → New Secret`,
> создать `HF_TOKEN` с правами read для доступа к моделям и датасету.

## 1. Setup

In [ ]:
!nvidia-smi

In [ ]:
import os, subprocess, pathlib, sys

REPO_URL  = "https://github.com/AStrateg2509/ocr_eval.git"
REPO_NAME = "ocr_eval"

if not pathlib.Path(REPO_NAME).exists():
    subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)

os.chdir(REPO_NAME)
sys.path.insert(0, str(pathlib.Path.cwd() / "src"))
print("CWD =", os.getcwd())

In [ ]:
!pip install -q -r requirements.txt
# poppler нужен olmOCR. На OmniDocBench anchor-текст отключён, но poppler
# всё равно нужен для импорта пакета.
!apt-get -qq install -y poppler-utils 2>&1 | tail -1

# Flash-attn (НЕ работает на Kaggle P100 — СКИПАЕМ блок).
# Раскомментируйте строку ниже только если запускаете на Ampere/Hopper/Ada (L4/A100/H100):
# !pip install --no-deps --force-reinstall https://github.com/Dao-AILab/flash-attention/releases/download/v2.7.1.post4/flash_attn-2.7.1.post4+cu12torch2.6cxx11abiFALSE-cp312-cp312-linux_x86_64.whl --no-build-isolation

In [ ]:
# Аутентификация HuggingFace через Kaggle Secrets.
# Вне Kaggle — задайте HF_TOKEN в переменных окружения.
import os
HF_TOKEN = None
try:
    from kaggle_secrets import UserSecretsClient
    HF_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    os.environ["HF_TOKEN"] = HF_TOKEN
    os.environ["HUGGING_FACE_HUB_TOKEN"] = HF_TOKEN
    print("HF token from Kaggle Secrets — OK")
except Exception as e:
    print(f"Kaggle Secrets unavailable ({e!r}); fallback на env")
    HF_TOKEN = os.environ.get("HF_TOKEN") or os.environ.get("HUGGING_FACE_HUB_TOKEN")

from huggingface_hub import login
if HF_TOKEN:
    login(token=HF_TOKEN, add_to_git_credential=False)
    print("logged in to HF")
else:
    print("ВНИМАНИЕ: токена нет — gated-модели не загрузятся")

## 2. Датасет: chunked-загрузка OmniDocBench

OmniDocBench содержит ~1653 страниц. HuggingFace Hub лимитирует ~1000 запросов в минуту с IP,
поэтому делим на два этапа:

1. **JSON-разметка** (несколько МБ) — одним вызовом.
2. **Изображения** — пачками по 200, с паузами 5 сек; после каждой пачки file-based verify.

Альтернативы (если этот путь окажется медленным):
* подключить OmniDocBench как Kaggle Dataset (один раз загрузить в Kaggle, дальше лежит локально без лимитов хаба);
* `git clone https://huggingface.co/datasets/opendatalab/OmniDocBench` (LFS);
* `datasets.load_dataset(с streaming=True)` и ленивый проход.

In [ ]:
from src.dataset_loader import (
    download_omnidocbench_json,
    list_repo_images, list_image_filenames,
    download_omnidocbench_images_chunked,
    verify_downloaded_images,
)

DATA_ROOT = "data/OmniDocBench"
download_omnidocbench_json(DATA_ROOT, token=HF_TOKEN)
print("JSON скачан:", list(pathlib.Path(DATA_ROOT).glob("OmniDocBench*.json")))

In [ ]:
# Чанк-загрузка. С chunk_size=200 и sleep=5s все ~1653 картинки загружаются за ~5–10 мин.
def progress(stage, **kw):
    print(f"[{stage}]", kw)

summary = download_omnidocbench_images_chunked(
    target_dir=DATA_ROOT,
    chunk_size=200,
    sleep_between_chunks=5.0,
    max_retries=5,
    token=HF_TOKEN,
    on_progress=progress,
)
summary

In [ ]:
# File-based верификация — пробегаем по диску, НЕ по списку из JSON.
expected = list_repo_images(token=HF_TOKEN)
report = verify_downloaded_images(DATA_ROOT, expected=expected)
print(f"present : {len(report['present'])}")
print(f"missing : {len(report['missing'])}")
print(f"broken  : {len(report['broken'])}")
if report['missing'][:5]:
    print("первые missing:", report['missing'][:5])
if report['broken'][:5]:
    print("первые broken :", report['broken'][:5])

# Если остались missing — повторяем загрузку (она идемпотентна).
if report['missing']:
    print("Докачиваем missing...")
    download_omnidocbench_images_chunked(DATA_ROOT, chunk_size=200,
                                         sleep_between_chunks=5.0, max_retries=5,
                                         token=HF_TOKEN)

## 3. Подвыборка: 100 страниц `academic_literature`, English

Один общий subset для всех трёх моделей — это даёт сопоставимые результаты.

In [ ]:
import json, pathlib
from src.dataset_loader import load_omnidocbench

items = load_omnidocbench(
    root=DATA_ROOT,
    page_types=["academic_literature"],
    languages=["english"],
    subset_size=100,
    seed=42,
    require_image_present=True,
)
print(f"отобрано: {len(items)} страниц")

subset_path = pathlib.Path("data/subset.json")
subset_path.parent.mkdir(parents=True, exist_ok=True)
subset_path.write_text(
    json.dumps([x.to_dict() for x in items], ensure_ascii=False),
    encoding="utf-8",
)
print("сохранено:", subset_path)

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

first = items[0]
img = Image.open(Path(DATA_ROOT) / first.image_path).convert("RGB")
fig, ax = plt.subplots(figsize=(7, 9))
ax.imshow(img); ax.axis("off")
ax.set_title(f"{first.page_id}  ·  {first.page_type}  ·  {first.language}")
plt.show()

## 4. Общие хелперы

In [ ]:
from src.utils import (
    load_config, JsonlWriter, Timer, cuda_free, gpu_info,
    already_processed_ids, read_jsonl,
)
from src.io_records import PredictionRecord
from src.dataset_loader import GroundTruth

import torch, traceback, json
from pathlib import Path
from PIL import Image

DATA_ROOT = Path("data/OmniDocBench")
subset = [GroundTruth(**rec) for rec in json.loads(
    Path("data/subset.json").read_text(encoding="utf-8"))]
print("subset:", len(subset), "страниц")
print("GPU:", gpu_info())

## 5. LightOnOCR-2-1B

Самая лёгкая модель — стартуем с неё, она гарантированно помещается в P100.

In [ ]:
from transformers import AutoModelForCausalLM, AutoModel, AutoProcessor

cfg = load_config("configs/lightonocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])

processor = AutoProcessor.from_pretrained(
    MODEL_REPO,
    trust_remote_code=cfg["model"]["trust_remote_code"],
    token=HF_TOKEN,
)
try:
    model = AutoModelForCausalLM.from_pretrained(
        MODEL_REPO,
        trust_remote_code=cfg["model"]["trust_remote_code"],
        torch_dtype=DTYPE,
        device_map=cfg["model"]["device_map"],
        token=HF_TOKEN,
    ).eval()
except Exception as e:
    print(f"AutoModelForCausalLM fail: {e!r}; fallback → AutoModel")
    model = AutoModel.from_pretrained(
        MODEL_REPO,
        trust_remote_code=cfg["model"]["trust_remote_code"],
        torch_dtype=DTYPE,
        device_map=cfg["model"]["device_map"],
        token=HF_TOKEN,
    ).eval()
print(gpu_info())

In [ ]:
out_path = Path(cfg["output"]["results_dir"]) / "predictions.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

PROMPT  = cfg["inference"]["prompt"]
MAX_NEW = cfg["inference"]["max_new_tokens"]

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="lightonocr")
        try:
            img = Image.open(img_path).convert("RGB")
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": PROMPT},
                ],
            }]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt")
            inputs = {k: v.to(model.device) for k, v in inputs.items()}
            with Timer("infer") as t, torch.no_grad():
                gen = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW,
                    do_sample=cfg["inference"]["do_sample"],
                    num_beams=cfg["inference"]["num_beams"],
                )
            input_len = inputs["input_ids"].shape[1]
            out = processor.batch_decode(gen[:, input_len:], skip_special_tokens=True)[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, processor; cuda_free(); print(gpu_info())

## 6. DeepSeek-OCR

Главный фикс прошлой версии: `model.infer(...)` НЕ возвращает markdown как строку —
он пишет файл `<output_path>/result.mmd`. Поэтому ставим `save_results=True` и читаем
результат с диска.

In [ ]:
from transformers import AutoModel, AutoTokenizer

cfg = load_config("configs/deepseek_ocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])

tokenizer = AutoTokenizer.from_pretrained(MODEL_REPO, trust_remote_code=True, token=HF_TOKEN)
model = AutoModel.from_pretrained(
    MODEL_REPO,
    trust_remote_code=True,
    torch_dtype=DTYPE,
    device_map=cfg["model"]["device_map"],
    token=HF_TOKEN,
).eval()
print(gpu_info())

In [ ]:
out_dir = Path(cfg["output"]["results_dir"])
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / "predictions.jsonl"
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

RESULT_FILENAME = cfg["inference"].get("result_filename", "result.mmd")

def _read_deepseek_result(folder: Path) -> str:
    candidates = [folder / RESULT_FILENAME,
                  folder / "result.mmd",
                  folder / "result.txt",
                  folder / "output.mmd"]
    for c in candidates:
        if c.exists() and c.stat().st_size > 0:
            return c.read_text(encoding="utf-8", errors="ignore")
    for f in folder.glob("*.mmd"):
        return f.read_text(encoding="utf-8", errors="ignore")
    return ""

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="deepseek_ocr")
        try:
            tmp_out = out_dir / gt.page_id
            tmp_out.mkdir(parents=True, exist_ok=True)
            with Timer("infer") as t:
                _ret = model.infer(
                    tokenizer,
                    prompt=cfg["inference"]["prompt"],
                    image_file=str(img_path),
                    output_path=str(tmp_out),
                    base_size=cfg["inference"]["base_size"],
                    image_size=cfg["inference"]["image_size"],
                    crop_mode=cfg["inference"]["crop_mode"],
                    save_results=cfg["inference"]["save_results"],
                    test_compress=cfg["inference"]["test_compress"],
                )
            text = _read_deepseek_result(tmp_out)
            if not text and isinstance(_ret, str):
                text = _ret
            rec.full_text = text
            rec.raw_output = text
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, tokenizer; cuda_free(); print(gpu_info())

## 7. olmOCR (Qwen2-VL-7B fine-tuned)

P100-friendly: `float16`, `attn_implementation="sdpa"`, уменьшенная картинка.
При OOM — урежьте `subset_size` / `max_new_tokens` / `max_image_long_side` в `configs/olmocr.yaml`.

In [ ]:
from transformers import Qwen2VLForConditionalGeneration, AutoProcessor

cfg = load_config("configs/olmocr.yaml")
MODEL_REPO = cfg["model"]["hf_repo"]
DTYPE = getattr(torch, cfg["model"]["torch_dtype"])

processor = AutoProcessor.from_pretrained(MODEL_REPO, token=HF_TOKEN)
model = Qwen2VLForConditionalGeneration.from_pretrained(
    MODEL_REPO,
    torch_dtype=DTYPE,
    device_map=cfg["model"]["device_map"],
    attn_implementation=cfg["model"]["attn_implementation"],
    low_cpu_mem_usage=cfg["model"]["low_cpu_mem_usage"],
    token=HF_TOKEN,
).eval()
print(gpu_info())

In [ ]:
out_path = Path(cfg["output"]["results_dir"]) / "predictions.jsonl"
out_path.parent.mkdir(parents=True, exist_ok=True)
done = already_processed_ids(out_path)
print(f"уже обработано: {len(done)} / {len(subset)}")

MAX_NEW   = cfg["inference"]["max_new_tokens"]
TEMP      = cfg["inference"]["temperature"]
LONG_SIDE = cfg["inference"]["max_image_long_side"]

OLMOCR_PROMPT = (
    "Below is the image of one page of a document. Just return the plain text "
    "representation of this document as if you were reading it naturally. "
    "Convert equations to LaTeX and tables to HTML. Do not hallucinate."
)

def _resize(img, long_side):
    w, h = img.size
    if max(w, h) <= long_side:
        return img
    if w >= h:
        new = (long_side, int(h * long_side / w))
    else:
        new = (int(w * long_side / h), long_side)
    return img.resize(new, Image.LANCZOS)

with JsonlWriter(out_path) as w:
    for gt in subset:
        if gt.page_id in done:
            continue
        img_path = DATA_ROOT / gt.image_path
        if not img_path.exists():
            continue
        rec = PredictionRecord(page_id=gt.page_id, model="olmocr")
        try:
            img = _resize(Image.open(img_path).convert("RGB"), LONG_SIDE)
            messages = [{
                "role": "user",
                "content": [
                    {"type": "image"},
                    {"type": "text", "text": OLMOCR_PROMPT},
                ],
            }]
            text = processor.apply_chat_template(
                messages, tokenize=False, add_generation_prompt=True,
            )
            inputs = processor(text=[text], images=[img], padding=True, return_tensors="pt").to(model.device)
            with Timer("infer") as t, torch.no_grad():
                gen = model.generate(
                    **inputs,
                    max_new_tokens=MAX_NEW,
                    do_sample=TEMP > 0,
                    temperature=TEMP if TEMP > 0 else 1.0,
                )
            out = processor.batch_decode(
                gen[:, inputs.input_ids.shape[1]:],
                skip_special_tokens=True,
            )[0]
            rec.full_text = out
            rec.raw_output = out
            rec.inference_time_s = t.elapsed
        except Exception as e:
            rec.error = f"{type(e).__name__}: {e}"
            traceback.print_exc()
        w.write(rec.to_dict())

print("→", out_path, "записей:", sum(1 for _ in open(out_path)))
del model, processor; cuda_free(); print(gpu_info())

## 8. Быстрая верификация выходов

Не метрики (это следующий шаг), а контроль того, что инференс реально записал данные.

In [ ]:
import pandas as pd

rows = []
for model_name in ("lightonocr", "deepseek_ocr", "olmocr"):
    p = Path("results") / model_name / "predictions.jsonl"
    if not p.exists():
        rows.append({"model": model_name, "records": 0, "errors": 0,
                     "avg_time_s": 0.0, "avg_text_len": 0})
        continue
    recs = read_jsonl(p)
    errors = sum(1 for r in recs if r.get("error"))
    times = [r["inference_time_s"] for r in recs if not r.get("error")]
    lens  = [len(r.get("full_text", "")) for r in recs if not r.get("error")]
    rows.append({
        "model": model_name,
        "records": len(recs),
        "errors": errors,
        "avg_time_s": round(sum(times) / max(1, len(times)), 2),
        "avg_text_len": round(sum(lens) / max(1, len(lens)), 0),
    })

pd.DataFrame(rows)

---
**Готово.** Дальше — модули метрик (Text / Table / IoU / Recall@5 / Overall)
и сводный CSV `results/summary.csv`. Это следующая порция кода.